In [ ]:
import json
import plotly.graph_objects as go
from datetime import datetime
import numpy as np
from flashalpha import FlashAlpha
from dotenv import load_dotenv
import os
from pathlib import Path

In [ ]:
load_dotenv(dotenv_path="../.env")

API_KEY = os.getenv("API_KEY")

TICKER = "TSLA"

EXPIRATION = "2026-06-01"

In [ ]:
FILE_PATH = Path("./data.json")

def get_options_chain():
    fa = FlashAlpha(API_KEY)
    return fa.gex(TICKER, expiration=EXPIRATION, min_oi=None)

# if FILE_PATH.exists():
#     data = json.loads(FILE_PATH.read_text())
#     print("Loaded from file")
# else:
#     data = get_options_chain()
#     FILE_PATH.write_text(json.dumps(data, indent=4))
#     print("Fetched and saved")


data = get_options_chain()

print(data)

In [ ]:
with open("./data.json", "r") as file:
    data = json.load(file)

In [ ]:
# Extract values
strikes = [x["strike"] for x in data["strikes"]]
call_gex = [x["call_gex"] for x in data["strikes"]]
put_gex = [x["put_gex"] for x in data["strikes"]]
net_gex = [x["net_gex"] for x in data["strikes"]]

underlying_price = data["underlying_price"]
gamma_flip = data["gamma_flip"]

# Y range for vertical lines
y_min = 0
# y_min = min(put_gex + net_gex) * 1.2
y_max = max(call_gex + net_gex + [1]) * 1.2

# Convert timestamp to human-readable format
ts = data['as_of']
dt = datetime.fromisoformat(ts.replace("Z", "+00:00"))
EXPIRATION = dt.strftime("%Y-%m-%d")

# Create figure
fig = go.Figure()

# Call GEX bars
fig.add_trace(
    go.Bar(
        x=strikes,
        y=call_gex,
        name="Call GEX",
        marker_color="#00ff99",
        visible=True
    )
)

# Put GEX bars
fig.add_trace(
    go.Bar(
        x=strikes,
        y=put_gex,
        name="Put GEX",
        marker_color="#ff4d4d",
        visible=True
    )
)

# Net GEX line
net_gex = np.array(net_gex)

# split values
net_gex_pos = np.where(net_gex > 0, net_gex, 0)
net_gex_neg = np.where(net_gex < 0, net_gex, 0)

# POSITIVE bars
fig.add_trace(
    go.Bar(
        x=strikes,
        y=net_gex_pos,
        name="Net GEX (+)",
        marker_color="#00ff99",
    )
)

# NEGATIVE bars
fig.add_trace(
    go.Bar(
        x=strikes,
        y=net_gex_neg,
        name="Net GEX (-)",
        marker_color="#ff4d4d",
    )
)

# UNDERLYING PRICE LINE (toggleable)
fig.add_trace(
    go.Scatter(
        x=[underlying_price, underlying_price],
        y=[y_min, y_max],
        mode="lines",
        name=f"Underlying ({underlying_price})",
        line=dict(
            color="yellow",
            width=2,
            dash="dash"
        ),
        hovertemplate=f"Underlying Price: {underlying_price}<extra></extra>"
    )
)

# GAMMA FLIP LINE (toggleable)
fig.add_trace(
    go.Scatter(
        x=[gamma_flip, gamma_flip],
        y=[y_min, y_max],
        mode="lines",
        name=f"Gamma Flip ({gamma_flip:.2f})",
        line=dict(
            color="orange",
            width=2,
            dash="dot"
        ),
        hovertemplate=f"Gamma Flip: {gamma_flip:.2f}<extra></extra>"
    )
)

fig.update_layout(
    title=dict(
        text=f"<b>{data['symbol']} GEX by Strike</b>  |  0DTE {EXPIRATION}  |  Spot ${underlying_price:.2f}",
        font_size=15, x=0.01,
    ),
    template="plotly_dark",
    barmode="relative",
    paper_bgcolor="black",
    plot_bgcolor="black",
    font=dict(color="white"),
    xaxis=dict(
        title="Strike",
        # showgrid=True,
        # gridcolor="#333",
        tickprefix="$",
        tickangle=60,
        # dtick=5
        tickmode="array",
        tickvals=strikes,
    ),
    yaxis=dict(
        title="Gamma Exposure",
        showgrid=True,
        gridcolor="#333"
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0)"
    ),
    hovermode="x unified",
    height=800,
)

# Show figure
fig.show()